In [11]:
import cv2
import numpy as np
import os
from matplotlib import pyplot as plt
import time
import mediapipe as mp
import json
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
from tqdm import tqdm
import torch.nn.functional as F

# Keypoints

In [12]:
mp_holistic = mp.solutions.holistic 
mp_drawing = mp.solutions.drawing_utils 

In [13]:
def mediapipe_detection(image, model):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image.flags.writeable = False                 
    results = model.process(image)                 
    image.flags.writeable = True              
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)  
    return image, results

In [14]:
def draw_landmarks(image, results):
    # mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION) 
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS) 
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS) 
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)

In [15]:
def draw_styled_landmarks(image, results):
    # mp_drawing.draw_landmarks(image, results.face_landmarks, mp_holistic.FACEMESH_TESSELATION, 
    #                          mp_drawing.DrawingSpec(color=(80,110,10), thickness=1, circle_radius=1), 
    #                          mp_drawing.DrawingSpec(color=(80,256,121), thickness=1, circle_radius=1)
    #                          ) 
    mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_holistic.POSE_CONNECTIONS,
                             mp_drawing.DrawingSpec(color=(80,22,10), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(80,44,121), thickness=2, circle_radius=2)
                             ) 
    mp_drawing.draw_landmarks(image, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(121,22,76), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(121,44,250), thickness=2, circle_radius=2)
                             ) 
    mp_drawing.draw_landmarks(image, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS, 
                             mp_drawing.DrawingSpec(color=(245,117,66), thickness=2, circle_radius=4), 
                             mp_drawing.DrawingSpec(color=(245,66,230), thickness=2, circle_radius=2)
                             ) 

# Extract Keypoint Values

In [66]:
def extract_keypoints(results):
    if results.pose_landmarks:
        pose_landmarks = np.array([[res.x, res.y, res.z]
                                   for res in results.pose_landmarks.landmark])
    else:
        pose_landmarks = np.zeros((33, 3))

    if results.left_hand_landmarks:
        lh_landmarks = np.array([[res.x, res.y, res.z]
                                 for res in results.left_hand_landmarks.landmark])
    else:
        lh_landmarks = np.zeros((21, 3))

    if results.right_hand_landmarks:
        rh_landmarks = np.array([[res.x, res.y, res.z]
                                 for res in results.right_hand_landmarks.landmark])
    else:
        rh_landmarks = np.zeros((21, 3))


    if np.any(pose_landmarks[23]) and np.any(pose_landmarks[24]):
        mid_hip = (pose_landmarks[23] + pose_landmarks[24]) / 2
    else:
        mid_hip = np.mean(pose_landmarks, axis=0)

    pose_landmarks -= mid_hip
    lh_landmarks -= mid_hip
    rh_landmarks -= mid_hip

    if np.any(pose_landmarks[11]) and np.any(pose_landmarks[23]):
        scale = np.linalg.norm(pose_landmarks[11] - pose_landmarks[23])
    else:
        scale = np.std(pose_landmarks) 

    scale = max(scale, 1e-6)
    pose_landmarks /= scale
    lh_landmarks /= scale
    rh_landmarks /= scale

    return np.concatenate([
        pose_landmarks.flatten(),
        lh_landmarks.flatten(),
        rh_landmarks.flatten()
    ])


# Folder Setup

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [41]:
WLASL_VID_PATH = os.path.join('D:/Work/text2sign/datasets/WLASL/videos')
WLASL_GLOSS_PATH = os.path.join('../../datasets/WLASL/WLASL_converted.json')

In [42]:
with open(WLASL_GLOSS_PATH, 'r') as f:
    wlasl_gloss = json.load(f)
    

In [1]:
import json

ISL_GIFS = os.path.join('../../datasets/ISL_Gifs')
ISL_GLOSS = os.path.join('../../datasets/invGlossList.json')

DATA_PATH = os.path.join('MP_Data_gloss') 
VIDEO_PATH = ISL_GIFS 
GLOSS_PATH = ISL_GLOSS
KEY_JSON_PATH = "test2keypoints_dataset.json"

with open(WLASL_GLOSS_PATH, 'r') as f:
    gloss = json.load(f)
    
actions = np.array(list(gloss.keys())[:100])
print(f"Actions: {actions} : {len(actions)}")

# 30 vids
SEQ_VID = 30

# Videos: 30 frames in length
SEQ_FRAMES = 30

STRT_FOLDER = 0

NameError: name 'os' is not defined

In [72]:
# for action in actions: 
#     if not os.path.exists(os.path.join(DATA_PATH, action)):
#         os.makedirs(os.path.join(DATA_PATH, action))
#     dirmax = np.max(np.array(os.listdir(os.path.join(DATA_PATH, action))).astype(int)) if os.listdir(os.path.join(DATA_PATH, action)) else 0
#     for sequence in range(1,no_sequences+1):
#         try: 
#             os.makedirs(os.path.join(DATA_PATH, action, str(dirmax+sequence)))
#         except:
#             pass


# Keypoint Value Collection

In [73]:
def extract_keypoints(results):
    if results.pose_landmarks:
        pose_landmarks = np.array([[res.x, res.y, res.z]
                                   for res in results.pose_landmarks.landmark])
    else:
        pose_landmarks = np.zeros((33, 3))

    if results.left_hand_landmarks:
        lh_landmarks = np.array([[res.x, res.y, res.z]
                                 for res in results.left_hand_landmarks.landmark])
    else:
        lh_landmarks = np.zeros((21, 3))

    if results.right_hand_landmarks:
        rh_landmarks = np.array([[res.x, res.y, res.z]
                                 for res in results.right_hand_landmarks.landmark])
    else:
        rh_landmarks = np.zeros((21, 3))


    if np.any(pose_landmarks[23]) and np.any(pose_landmarks[24]):
        mid_hip = (pose_landmarks[23] + pose_landmarks[24]) / 2
    else:
        mid_hip = np.mean(pose_landmarks, axis=0)

    pose_landmarks -= mid_hip
    lh_landmarks -= mid_hip
    rh_landmarks -= mid_hip

    if np.any(pose_landmarks[11]) and np.any(pose_landmarks[23]):
        scale = np.linalg.norm(pose_landmarks[11] - pose_landmarks[23])
    else:
        scale = np.std(pose_landmarks) 

    scale = max(scale, 1e-6)
    pose_landmarks /= scale
    lh_landmarks /= scale
    rh_landmarks /= scale

    return np.concatenate([
        pose_landmarks.flatten(),
        lh_landmarks.flatten(),
        rh_landmarks.flatten()
    ])


In [75]:
mp_holistic   = mp.solutions.holistic
mp_drawing    = mp.solutions.drawing_utils
mp_pose_conn  = mp_holistic.POSE_CONNECTIONS
mp_hand_conn  = mp_holistic.HAND_CONNECTIONS

def collect_keypoints():
    skips = 0
    tots  = 0

    with mp_holistic.Holistic(min_detection_confidence=0.5,
                              min_tracking_confidence=0.5) as holistic:

        for action_idx, action in enumerate(actions):
            vids = gloss.get(action, [])
            for seq, vid_id in enumerate(vids):
                tots += 1
                video_path = os.path.join(WLASL_VID_PATH, vid_id + ".mp4")
                if not os.path.isfile(video_path):
                    skips += 1
                    print(f"[!] Missing file, skipping: {video_path}")
                    continue

                cap = cv2.VideoCapture(video_path)
                frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
                if frame_count < SEQ_FRAMES:
                    skips += 1
                    print(f"[!] Video too short ({frame_count} frames), skipping: {video_path}")
                    cap.release()
                    continue
                
                INFO = f"[>] [{action_idx+1}/{len(actions)}] Action={action}, Video={vid_id}, Frames={frame_count}"
                print(INFO)

                

                frame_indices = np.linspace(
                    0, frame_count-1, SEQ_FRAMES, dtype=int
                )

                for frame_num, idx in enumerate(frame_indices):
                    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
                    ret, frame = cap.read()
                    if not ret:
                        print(f"[!] Frame read failed at idx={idx}")
                        continue

                    frame = cv2.resize(frame, (512, 512))
                    image, results = mediapipe_detection(frame, holistic)

                    mp_drawing.draw_landmarks(
                        image, results.pose_landmarks, mp_pose_conn,
                        mp_drawing.DrawingSpec((0,255,0), 2, 1),
                        mp_drawing.DrawingSpec((0,128,0), 2)
                    )
                    mp_drawing.draw_landmarks(
                        image, results.left_hand_landmarks, mp_hand_conn,
                        mp_drawing.DrawingSpec((255,0,0), 2, 1),
                        mp_drawing.DrawingSpec((128,0,0), 2)
                    )
                    mp_drawing.draw_landmarks(
                        image, results.right_hand_landmarks, mp_hand_conn,
                        mp_drawing.DrawingSpec((0,0,255), 2, 0),
                        mp_drawing.DrawingSpec((0,0,128), 2)
                    )

                    if frame_num == 0:
                        cv2.imshow("Collecting", image)
                        cv2.waitKey(300)
                    else:
                        cv2.imshow("Collecting", image)
                        cv2.waitKey(1)

                    keypoints = extract_keypoints(results)
                    out_dir = os.path.join(DATA_PATH, action, str(seq))
                    os.makedirs(out_dir, exist_ok=True)
                    save_path = os.path.join(out_dir, f"{frame_num}.npy")
                    # os.makedirs(save_path, exist_ok=True) 
                    np.save(save_path, keypoints)

                cap.release()

    cv2.destroyAllWindows()
    print(f"[Done] Total videos: {tots}, Skipped: {skips}")

if __name__ == "__main__":
    collect_keypoints()


[!] Missing file, skipping: D:/Work/text2sign/datasets/WLASL/videos\57519.mp4
[>] [1/100] Action=test, Video=57530, Frames=81
[>] [1/100] Action=test, Video=57531, Frames=61
[!] Missing file, skipping: D:/Work/text2sign/datasets/WLASL/videos\57532.mp4
[!] Missing file, skipping: D:/Work/text2sign/datasets/WLASL/videos\57533.mp4
[>] [1/100] Action=test, Video=57534, Frames=45
[!] Frame read failed at idx=44
[>] [1/100] Action=test, Video=57535, Frames=47
[!] Frame read failed at idx=44
[!] Frame read failed at idx=46
[!] Missing file, skipping: D:/Work/text2sign/datasets/WLASL/videos\57536.mp4
[>] [1/100] Action=test, Video=57527, Frames=97
[!] Missing file, skipping: D:/Work/text2sign/datasets/WLASL/videos\67290.mp4
[!] Missing file, skipping: D:/Work/text2sign/datasets/WLASL/videos\57537.mp4
[!] Missing file, skipping: D:/Work/text2sign/datasets/WLASL/videos\57539.mp4
[!] Missing file, skipping: D:/Work/text2sign/datasets/WLASL/videos\57540.mp4
[>] [1/100] Action=test, Video=57541, Fr

# Preprocess, Labels and Feats

In [76]:
def pad_sequences(data_path, expected_len=31):
    print("[*] Padding incomplete sequences...")

    for action in tqdm(os.listdir(data_path)):
        action_path = os.path.join(data_path, action)
        if not os.path.isdir(action_path):
            continue

        for seq in os.listdir(action_path):
            seq_path = os.path.join(action_path, seq)
            if not os.path.isdir(seq_path):
                continue

            files = sorted([f for f in os.listdir(seq_path) if f.endswith(".npy")], key=lambda x: int(x.split(".")[0]))
            current_len = len(files)

            if current_len < expected_len:
                last_frame = np.load(os.path.join(seq_path, files[-1]))
                for i in range(current_len, expected_len):
                    np.save(os.path.join(seq_path, f"{i}.npy"), last_frame)
                print(f"[+] Padded: {action}/{seq} (had {current_len}, now {expected_len})")

    print("[*] Padding complete.")

if __name__ == "__main__":
    pad_sequences(DATA_PATH)


[*] Padding incomplete sequences...


  0%|          | 0/100 [00:00<?, ?it/s]

[+] Padded: accident/1 (had 30, now 31)
[+] Padded: accident/10 (had 30, now 31)
[+] Padded: accident/11 (had 30, now 31)


  1%|          | 1/100 [00:00<00:40,  2.47it/s]

[+] Padded: accident/12 (had 30, now 31)
[+] Padded: accident/15 (had 29, now 31)
[+] Padded: accident/2 (had 30, now 31)
[+] Padded: accident/3 (had 30, now 31)
[+] Padded: accident/4 (had 30, now 31)
[+] Padded: accident/5 (had 30, now 31)
[+] Padded: accident/6 (had 30, now 31)
[+] Padded: accident/7 (had 30, now 31)
[+] Padded: accident/9 (had 29, now 31)
[+] Padded: africa/1 (had 30, now 31)
[+] Padded: africa/17 (had 30, now 31)
[+] Padded: africa/2 (had 30, now 31)


...

[+] Padded: wrong/4 (had 30, now 31)
[+] Padded: wrong/5 (had 30, now 31)
[+] Padded: wrong/8 (had 29, now 31)
[+] Padded: year/1 (had 30, now 31)
[+] Padded: year/10 (had 30, now 31)
[+] Padded: year/11 (had 30, now 31)
[+] Padded: year/12 (had 30, now 31)
[+] Padded: year/13 (had 30, now 31)
[+] Padded: year/19 (had 29, now 31)
[+] Padded: year/7 (had 30, now 31)
[+] Padded: year/8 (had 30, now 31)
[+] Padded: year/9 (had 30, now 31)


 99%|█████████▉| 99/100 [01:12<00:00,  4.74it/s]

[+] Padded: yes/1 (had 30, now 31)
[+] Padded: yes/12 (had 30, now 31)
[+] Padded: yes/14 (had 30, now 31)
[+] Padded: yes/17 (had 30, now 31)
[+] Padded: yes/18 (had 30, now 31)


100%|██████████| 100/100 [01:13<00:00,  1.37it/s]

[+] Padded: yes/19 (had 30, now 31)
[+] Padded: yes/2 (had 30, now 31)
[+] Padded: yes/5 (had 30, now 31)
[+] Padded: yes/6 (had 29, now 31)
[+] Padded: yes/7 (had 29, now 31)
[+] Padded: yes/8 (had 29, now 31)
[*] Padding complete.


In [ ]:
dataset_entries = []

for action in actions:
    action_dir = os.path.join(DATA_PATH, action)
    if not os.path.exists(action_dir):
        print(f"[!] Missing action dir: {action_dir}")
        continue
    
    for sequence in sorted(os.listdir(action_dir), key = lambda x: int(x)):
        sequence_path = os.path.join(action_dir, sequence)
        if not os.path.isdir(sequence_path):
            print(f"[!] Missing Sequence: {sequence_path}")
            continue
        
        frame_files = [f for f in os.listdir(sequence_path) if f.endswith('.npy')]
        if len(frame_files) < 30:
            print(f"[!] Skipping Incomplete Sequence: {action}/{sequence} (only {len(frame_files)} frames)")
            continue
        
        dataset_entries.append({
            "text": action,
            "keypoints_path": sequence_path.replace("\\", "/")
        })
        
output_path = "test2keypoints_dataset.json"
with open(output_path, 'w') as f:
    json.dump(dataset_entries, f, indent=2)
    
print(f"[+] Dataset length: {len(dataset_entries)}")

[+] Dataset length: 966


In [83]:
zero_count = 0
total_count = 0

for entry in dataset_entries:
    kp_dir = entry["keypoints_path"]
    frame_files = [f for f in os.listdir(kp_dir) if f.endswith('.npy')]
    for f_name in frame_files:
        arr = np.load(os.path.join(kp_dir, f_name))
        total_count += 1
        if np.all(arr == 0):
            zero_count += 1

print(f"Total .npy files: {total_count}")
print(f".npy files with only zeroes: {zero_count}")


Total .npy files: 29946
.npy files with only zeroes: 266


In [78]:
label_map = {label:num for num, label in enumerate(actions)}
label_map

{'test': 0,
 'book': 1,
 'drink': 2,
 'computer': 3,
 'before': 4,
 'chair': 5,
 'go': 6,
 'clothes': 7,
 'who': 8,
 'candy': 9,
 'cousin': 10,
 'deaf': 11,
 'fine': 12,
 'help': 13,
 'no': 14,
 'thin': 15,
 'walk': 16,
 'year': 17,
 'yes': 18,
 'all': 19,
 'black': 20,
 'cool': 21,
 'finish': 22,
 'hot': 23,
 'like': 24,
 'many': 25,
 'mother': 26,
 'now': 27,
 'orange': 28,
 'table': 29,
 'thanksgiving': 30,
 'what': 31,
 'woman': 32,
 'bed': 33,
 'blue': 34,
 'bowling': 35,
 'can': 36,
 'dog': 37,
 'family': 38,
 'fish': 39,
 'graduate': 40,
 'hat': 41,
 'hearing': 42,
 'kiss': 43,
 'language': 44,
 'later': 45,
 'man': 46,
 'shirt': 47,
 'study': 48,
 'tall': 49,
 'white': 50,
 'wrong': 51,
 'accident': 52,
 'apple': 53,
 'bird': 54,
 'change': 55,
 'color': 56,
 'corn': 57,
 'cow': 58,
 'dance': 59,
 'dark': 60,
 'doctor': 61,
 'eat': 62,
 'enjoy': 63,
 'forget': 64,
 'give': 65,
 'last': 66,
 'meet': 67,
 'pink': 68,
 'pizza': 69,
 'play': 70,
 'school': 71,
 'secretary': 72,
 's

In [79]:
class T2SDataset(Dataset):
    def __init__(self, json_path, sequence_length=30, feature_dim=1662, max_text_len=10):
        with open(json_path, 'r') as f:
            self.data = json.load(f)

        self.sequence_length = sequence_length
        self.feature_dim = feature_dim
        self.max_text_len = max_text_len

        all_text = [entry['text'] for entry in self.data]
        vocab_words = set(word for t in all_text for word in t.lower().split())
        self.vocab = {'<PAD>': 0, '<UNK>': 1}
        self.vocab.update({word: idx+2 for idx, word in enumerate(sorted(vocab_words))})

    def __len__(self):
        return len(self.data)

    def text_to_tensor(self, text):
        tokens = text.lower().split()
        ids = [self.vocab.get(tok, self.vocab['<UNK>']) for tok in tokens]
        ids += [self.vocab['<PAD>']] * (self.max_text_len - len(ids))
        return torch.tensor(ids[:self.max_text_len])

    def __getitem__(self, idx):
        item = self.data[idx]
        x = self.text_to_tensor(item["text"])
        y = []

        for i in range(self.sequence_length):
            path = os.path.join(item["keypoints_path"], f"{i}.npy")
            if os.path.exists(path):
                kp = np.load(path)
            else:
                kp = y[-1] if y else np.zeros(self.feature_dim)
            y.append(kp)
        
        y = np.stack(y)
        return x, torch.tensor(y, dtype=torch.float32)

# LSTM Model Arch

## LSTM Model

In [80]:
class LSTM_t2s(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, num_layers=1):
        super(LSTM_t2s, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim * 30)
        self.output_dim = output_dim

    def forward(self, x):
        embedded = self.embedding(x)  # (B, T, D)
        _, (hn, _) = self.lstm(embedded)
        output = self.fc(hn[-1])
        return output.view(-1, 30, self.output_dim)

## Transformer LSTM

In [84]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)                # (max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)   # (max_len, 1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)       
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))       # (1, max_len, d_model)

    def forward(self, x):
        # x: (B, T, d_model)
        x = x + self.pe[:, :x.size(1)]
        return x

class Text2SignHybrid(nn.Module):
    def __init__(self,
                 vocab_size: int,
                 embed_dim: int,
                 trans_dim: int,
                 lstm_hidden: int,
                 num_layers: int,
                 output_dim: int,
                 seq_len: int = 30,
                 num_heads: int = 4,
                 tcn_channels: list = [128, 64]):
        super().__init__()
        self.seq_len = seq_len
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model=embed_dim, max_len=seq_len)
        
        # ----- Transformer Encoder --------
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead= num_heads,
            dim_feedforward=trans_dim,
            dropout=0.1,
            activation='relu'
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=2)
        
        # --------- LSTM Decoder ------------
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=lstm_hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.1
        )
        
        # -------------- Temporal Convolutional Refinement ---------------
        tcn_layers = []
        in_ch = lstm_hidden
        for out_ch in tcn_channels:
            tcn_layers += [
                nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.BatchNorm1d(out_ch)
            ]
            in_ch = out_ch
        self.tcn = nn.Sequential(*tcn_layers)
        
        # ---- Final projection ---
        self.fc = nn.Linear(tcn_channels[-1], output_dim)
        

    def forward(self, x):
        B = x.size(0)
        
        emb = self.embedding(x)                # (B, T_text, embed_dim)
        emb = self.pos_enc(emb)                # "
        
        tr_in = emb.permute(1,0,2)             # (T_text, B, D)
        tr_out = self.transformer(tr_in)       # (T_text, B, D)
        tr_out = tr_out.permute(1,0,2)         # (B, T_text, D)
        

        token_ctx = tr_out[:, -1, :].unsqueeze(1)  # (B,1,D)
        dec_input = token_ctx.repeat(1, self.seq_len, 1)  # (B, seq_len, D)
        
        lstm_out, _ = self.lstm(dec_input)     # (B, seq_len, lstm_hidden)
        
        #  (B, C, T)
        tcn_in = lstm_out.permute(0,2,1)      # (B, lstm_hidden, seq_len)
        tcn_out = self.tcn(tcn_in)             # (B, last_ch, seq_len)
        
        tcn_out = tcn_out.permute(0,2,1)       # (B, seq_len, last_ch)
        # out   = self.fc(tcn_out)             # (B, seq_len, output_dim)
        out = torch.sigmoid(self.fc(tcn_out))

        return out


# Training Setup

In [85]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
def train_model(json_path, num_epochs=100, batch_size=8, lr=1e-3):
    dataset = T2SDataset(json_path)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=0)

    # model = LSTM_t2s(
    #     vocab_size=len(dataset.vocab),
    #     embed_dim=128,
    #     hidden_dim=512,
    #     output_dim=1662
    # )
    
    model = Text2SignHybrid(
        vocab_size  = len(dataset.vocab),
        output_dim  = 225,        
        seq_len     = 30,          
        embed_dim   = 128,        
        trans_dim   = 256,         
        lstm_hidden = 512,         
        num_layers  = 2,           
        num_heads   = 4,           
        tcn_channels= [128, 64]
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[=] Device: {device}")
    model.to(device)

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            y_pred = model(x)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{num_epochs} - Loss: {running_loss / len(dataloader):.4f}")

    torch.save(model.state_dict(), "text2sign_model.pth")
    np.save("alt_model_vocab.npy", dataset.vocab)
    print("[+] Model saved to text2sign_model.pth")
    return model, dataset.vocab


In [86]:
MODEL, VOCAB = train_model(KEY_JSON_PATH)

[=] Device: cuda
Epoch 1/100 - Loss: 1.4639
Epoch 2/100 - Loss: 1.0581
Epoch 3/100 - Loss: 1.0189
Epoch 4/100 - Loss: 0.9976
Epoch 5/100 - Loss: 0.9767
Epoch 6/100 - Loss: 0.9756
Epoch 7/100 - Loss: 0.9744
Epoch 8/100 - Loss: 0.9740
Epoch 9/100 - Loss: 0.9738
Epoch 10/100 - Loss: 0.9736
Epoch 11/100 - Loss: 0.9735
Epoch 12/100 - Loss: 0.9732
Epoch 13/100 - Loss: 0.9730
Epoch 14/100 - Loss: 0.9728
Epoch 15/100 - Loss: 0.9730
Epoch 16/100 - Loss: 0.9728
Epoch 17/100 - Loss: 0.9731
Epoch 18/100 - Loss: 0.9728
Epoch 19/100 - Loss: 0.9729
Epoch 20/100 - Loss: 0.9729
Epoch 21/100 - Loss: 0.9727
Epoch 22/100 - Loss: 0.9729
Epoch 23/100 - Loss: 0.9726
Epoch 24/100 - Loss: 0.9728
Epoch 25/100 - Loss: 0.9723
Epoch 26/100 - Loss: 0.9729
Epoch 27/100 - Loss: 0.9725
Epoch 28/100 - Loss: 0.9724
Epoch 29/100 - Loss: 0.9724
Epoch 30/100 - Loss: 0.9723
Epoch 31/100 - Loss: 0.9552
Epoch 32/100 - Loss: 0.9414
Epoch 33/100 - Loss: 0.9411
Epoch 34/100 - Loss: 0.9413
Epoch 35/100 - Loss: 0.9411
Epoch 36/100

In [87]:
np.save("alt_model_vocab.npy", VOCAB)